# Phase 2: Sensor Preprocessing, Synchronization & Normalization

**Objective**: Synchronize multi-modal IMU (~10 Hz) and GPS (~1 Hz) streams, filter sensor high-frequency noise with a 4th-order Butterworth filter, clip outliers, and establish a leak-free normalization pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing import (
    load_accelerometers,
    load_gps,
    synchronise_sensors,
    butterworth_lowpass,
    clip_outliers_iqr,
    handle_missing_values,
    SensorNormaliser,
)

%matplotlib inline

## 1. Sensor Synchronization
GPS (~1 Hz) is linearly interpolated onto IMU (~10 Hz) timestamps, attaching GPS speed and coordinates to every inertial measurement.

In [ ]:
trip_dir = Path('../data/raw/UAH-DriveSet/D1/20151110175712-16km-D1-NORMAL1-SECONDARY')
raw_imu = load_accelerometers(trip_dir / 'RAW_ACCELEROMETERS.txt')
raw_gps = load_gps(trip_dir / 'RAW_GPS.txt')

print(f"Raw IMU rows: {len(raw_imu)}, Raw GPS rows: {len(raw_gps)}")
synced = synchronise_sensors(raw_imu, raw_gps)
print(f"Synchronized rows: {len(synced)}")
display(synced[['timestamp', 'ax_filt', 'ay_filt', 'speed_kmh', 'latitude', 'longitude']].head())

## 2. Low-Pass Butterworth Filtering (2.0 Hz Cutoff)

In [ ]:
t = synced['timestamp'].values[:300]
raw_signal = synced['ax_raw'].values[:300]
filtered_signal = butterworth_lowpass(raw_signal, cutoff_hz=2.0, fs_hz=10.0)

plt.figure(figsize=(11, 4))
plt.plot(t, raw_signal, label="Raw Sensor Signal (ax_raw)", alpha=0.5, color="gray")
plt.plot(t, filtered_signal, label="Butterworth Filtered (2.0 Hz)", color="#2980b9", lw=1.5)
plt.title("Sensor Low-Pass Butterworth Noise Filtering", fontweight="bold")
plt.xlabel("Time (s)")
plt.ylabel("Acceleration (g)")
plt.legend()
plt.show()

## 3. Data Leakage-Free Normalization
`SensorNormaliser` fits only on training trips and transforms unseen validation and test trips.

In [ ]:
normaliser = SensorNormaliser(columns=['ax_filt', 'ay_filt', 'az_filt', 'speed_kmh'])
train_sample = synced.iloc[:400].copy()
test_sample = synced.iloc[400:].copy()

train_norm = normaliser.fit_transform(train_sample)
test_norm = normaliser.transform(test_sample)

print(f"Train normalized mean: {train_norm['ax_filt'].mean():.4f}, std: {train_norm['ax_filt'].std():.4f}")
print(f"Test transformed mean: {test_norm['ax_filt'].mean():.4f}, std: {test_norm['ax_filt'].std():.4f}")